# Programme 4 — Jacobi et Gauss-Seidel

**Référence :** Kröger, *Numerische Mathematik 1 für AMP*, section 2.5.

Ce notebook accompagne `jacobi_gauss_seidel.py` et illustre :

1. Pourquoi on a besoin de méthodes itératives (motivation Beispiel 2.24)
2. Les formules de **Jacobi** (2.17) et **Gauss-Seidel** (2.18)
3. La reproduction de l'**Übung 2.26**
4. Les critères de convergence : **dominance diagonale** (Satz 2.37) et **rayon spectral** (Satz 2.35)
5. Application à l'équation de la chaleur 2D (Beispiel 2.24)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from jacobi_gauss_seidel import (
    jacobi, gauss_seidel,
    matrice_iteration_jacobi, matrice_iteration_gauss_seidel,
    rayon_spectral, est_strictement_diag_dominant, diagnostiquer,
    systeme_chaleur_2d,
    tracer_convergence, tracer_solution_chaleur,
)

## 1. Motivation — pourquoi pas Gauss ?

Le **Beispiel 2.24** du script considère l'équation de la chaleur sur une grille 1000×1000. Cela donne un système $1\,002\,001 \times 1\,002\,001$ avec **5 entrées non nulles par ligne** seulement.

Estimations du script (PC ~$10^{11}$ ops/s) :

| Approche | Temps | Mémoire |
|---|---|---|
| Gauss naïf ($\frac{1}{3}n^3$ ops) | ~1 mois | 8 To |
| Gauss avec exploitation de la bande | ~17 min | 1,5 Go |
| Méthode itérative (1 step = $m \cdot n$) | quelques secondes/step | quelques Mo |

**Idée :** abandonner l'exactitude. Au lieu de calculer $x^* = A^{-1}b$, construire une suite $x_0, x_1, x_2, \dots$ qui converge vers $x^*$. Chaque étape est très peu coûteuse, et on s'arrête quand on a la précision voulue.

## 2. Les deux méthodes

**Idée commune :** isoler $x_i$ dans la $i$-ème équation :
$$x_i = \frac{1}{a_{ii}} \left( b_i - \sum_{j \neq i} a_{ij} x_j \right)$$

**Jacobi (formule 2.17, Gesamtschritt) :** on utilise les anciennes valeurs partout.
$$x_i^+ = \frac{1}{a_{ii}} \left( b_i - \sum_{j \neq i} a_{ij} x_j \right)$$

**Gauss-Seidel (formule 2.18, Einzelschritt) :** on utilise les nouvelles valeurs déjà calculées pour $j < i$.
$$x_i^+ = \frac{1}{a_{ii}} \left( b_i - \sum_{j<i} a_{ij} x_j^+ - \sum_{j>i} a_{ij} x_j \right)$$

Intuitivement, Gauss-Seidel devrait être meilleur : on injecte de l'information plus fraîche.

## 3. Übung 2.26 — système 3×3

$$A = \begin{pmatrix} 5 & 1 & -1 \\ 3 & -10 & 2 \\ 1 & -2 & 5 \end{pmatrix}, \quad b = \begin{pmatrix} 9 \\ 8 \\ 7 \end{pmatrix}, \quad x_0 = (1, 1, 1)^T$$

In [ ]:
A = np.array([[5, 1, -1], [3, -10, 2], [1, -2, 5]], dtype=float)
b = np.array([9, 8, 7], dtype=float)
x0 = np.array([1, 1, 1], dtype=float)

diag = diagnostiquer(A)
print(diag)

La matrice est strictement diagonalement dominante (Def. 2.36), donc par le **Satz 2.37** les deux méthodes convergent quel que soit $x_0$.

Les rayons spectraux $\rho(S_J) \approx 0.33$ et $\rho(S_{GS}) \approx 0.09$ donnent les rates de convergence asymptotique :
- Jacobi : $-\log_{10}(0.33) \approx 0.48$ chiffres décimaux par itération
- Gauss-Seidel : $-\log_{10}(0.09) \approx 1.04$ chiffres par itération

Donc Gauss-Seidel devrait être ~ **2× plus rapide**.

In [ ]:
res_j = jacobi(A, b, x0=x0, tol=1e-12)
res_gs = gauss_seidel(A, b, x0=x0, tol=1e-12)

print(f'Solution exacte (numpy) : {np.linalg.solve(A, b)}')
print(f'Jacobi       : {res_j.iterations} itérations -> x = {res_j.x}')
print(f'Gauss-Seidel : {res_gs.iterations} itérations -> x = {res_gs.x}')
print(f'\nRatio empirique J/GS : {res_j.iterations / res_gs.iterations:.2f}× (théorique : 2×)')

In [ ]:
tracer_convergence([res_j, res_gs])
plt.show()

On voit clairement les deux droites en échelle semi-log — c'est la signature d'une **convergence linéaire** (Definition 2.31). La pente de chaque droite est $-\log_{10}(\rho(S))$.

## 4. Quand ça ne converge pas

Pas de garantie sans hypothèse. Si on permute les lignes du système précédent :

In [ ]:
# Système permuté — plus du tout diagonalement dominant
A_bad = np.array([[1, -2, 5], [5, 1, -1], [3, -10, 2]], dtype=float)
b_bad = np.array([7, 9, 8], dtype=float)

diag_bad = diagnostiquer(A_bad)
print(diag_bad)

In [ ]:
res_bad = jacobi(A_bad, b_bad, tol=1e-10, n_max=100)
print(f'Jacobi : {res_bad.iterations} itérations, convergé = {res_bad.converge}')
print(f'Résidu final : {res_bad.historique_residus[-1]:.2e}  (=> diverge)')

$\rho(S_J) > 1$ donc divergence garantie (Satz 2.35). En pratique, il faut souvent réordonner les équations ou utiliser un préconditionneur.

## 5. Beispiel 2.24 — équation de la chaleur 2D

On résout $-\Delta T = 0$ sur le carré unité avec :
- Bord du haut : $T = 100$ ("source chaude")
- Trois autres bords : $T = 0$

Discrétisation par schéma à 5 points : pour chaque point intérieur,
$$4 T_{i,j} - T_{i\pm 1, j} - T_{i, j\pm 1} = 0$$

Pour une grille $n \times n$ on obtient un système creux $n^2 \times n^2$ avec **5 entrées non nulles par ligne**.

In [ ]:
n = 30
A_h, b_h = systeme_chaleur_2d(n, bord_haut=100.0)
diag_h = diagnostiquer(A_h)
print(f'Système {n*n}×{n*n} avec 5 non-zéros par ligne')
print(diag_h)

In [ ]:
res_j = jacobi(A_h, b_h, tol=1e-6, n_max=50_000)
res_gs = gauss_seidel(A_h, b_h, tol=1e-6, n_max=50_000)

print(f'Jacobi       : {res_j.iterations} itérations')
print(f'Gauss-Seidel : {res_gs.iterations} itérations')
print(f'Ratio empirique : {res_j.iterations / res_gs.iterations:.2f}× (théorique : 2×)')

Le rapport ~2× entre Jacobi et Gauss-Seidel est une propriété caractéristique des **matrices Stieltjes** (positives définies, hors-diagonale ≤ 0) — précisément le cas du laplacien discret. On le retrouve à toutes les tailles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tracer_convergence([res_j, res_gs], ax=axes[0])
tracer_solution_chaleur(res_gs.x, n, ax=axes[1])
plt.tight_layout()
plt.show()

## 6. Comportement en fonction de la taille

Pour le laplacien sur grille $n \times n$, on peut montrer que $\rho(S_J) = \cos(\pi h)$ avec $h = 1/(n+1)$. Donc $\rho \to 1$ quand $n \to \infty$ — la convergence ralentit terriblement.

C'est la motivation pour des méthodes plus avancées : **SOR**, **gradient conjugué**, **multigrille** (mentionnées dans l'Ausblick 2.5.11 du script).

In [ ]:
print(f'{"n":>4} | {"taille":>8} | {"ρ(S_J)":>10} | {"iter Jacobi":>12} | {"iter GS":>10}')
print('-' * 60)
for n in [5, 10, 20, 30]:
    A_h, b_h = systeme_chaleur_2d(n, bord_haut=100.0)
    diag = diagnostiquer(A_h)
    rj = jacobi(A_h, b_h, tol=1e-6, n_max=50_000)
    rgs = gauss_seidel(A_h, b_h, tol=1e-6, n_max=50_000)
    print(f'{n:>4} | {n*n:>8} | {diag.rho_jacobi:>10.6f} | {rj.iterations:>12} | {rgs.iterations:>10}')

On observe que le nombre d'itérations augmente comme $O(n^2)$ : doubler $n$ multiplie le nombre d'itérations par 4. C'est un défaut majeur des méthodes itératives "classiques" sur les EDP elliptiques.

## Récapitulatif (Satz 2.27, 2.35, 2.37)

| Aspect | Jacobi | Gauss-Seidel |
|---|---|---|
| Coût par itération | $m \cdot n$ | $m \cdot n$ |
| Vectorisable ? | oui | non (séquentiel) |
| Parallélisable ? | trivialement | non (sauf red-black) |
| Convergence si diag. dom. stricte | oui (Satz 2.37) | oui aussi |
| Vitesse pour matrices Stieltjes | référence | ~2× plus rapide |

**Prochain programme :** `newton_scalar.py` — sortir du linéaire avec Newton 1D, mesurer expérimentalement l'ordre de convergence quadratique.